# Notebook 04 — SHAP Analysis
**Part A — Temporal ML interpretability**

## What this notebook does
1. Trains a Random Forest on each of the 4 bioaerosol targets  
2. Computes **SHAP (SHapley Additive exPlanations)** values using TreeExplainer  
3. Produces:
   - SHAP **bar plot** (mean absolute SHAP — overall importance)  
   - SHAP **beeswarm plot** (directional effects by feature value)  
4. Prints the key directional findings

## What SHAP tells you
Unlike MDI or permutation importance, SHAP shows **direction**:  
- Does high wind speed *increase* or *decrease* the prediction?  
- At what feature values does the effect change sign?

## Important framing
LOO-CV R² is negative for all targets (see Notebook 03).  
SHAP values here represent **directional feature associations** within the model,  
not predictive accuracy. They are valid and interpretable despite negative R².

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP — run this cell first if using Google Colab
# ═══════════════════════════════════════════════════════════════
import os, sys

# Option A: Clone the GitHub repo directly in Colab (recommended)
# !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git
# os.chdir('geoai-bioaerosol-prediction')

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

In [ ]:
# ── Install (uncomment on Colab) ──────────────────────────────────
# !pip install shap scikit-learn matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import shap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/df_analysis.csv')
for col in ['Aspergillus_conc', 'Alternaria_conc']:
    df[col] = df[col].fillna(df[col].median())

MET_FEATURES = ['GHI', 'Tamb', 'RH', 'WS', 'BP', 'sin_WD', 'cos_WD']
FEAT_LABELS  = ['GHI', 'Temp', 'RH', 'Wind Speed', 'Pressure', 'sin(WD)', 'cos(WD)']
TARGETS = {
    'pollen_conc':      ('Pollen',       '#378ADD'),
    'fungus_conc':      ('Total Fungus', '#D85A30'),
    'Aspergillus_conc': ('Aspergillus',  '#1D9E75'),
    'Alternaria_conc':  ('Alternaria',   '#BA7517'),
}
X = df[MET_FEATURES].values
print('SHAP ready. n =', len(df))

In [ ]:
# ── Compute SHAP values for all 4 targets ─────────────────────────
shap_store = {}

for tcol, (tlabel, col) in TARGETS.items():
    y    = df[tcol].values
    mask = ~np.isnan(y)
    Xm, ym = X[mask], y[mask]

    # Train RF on full dataset
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(Xm, ym)

    # TreeExplainer gives exact Shapley values for tree models
    explainer = shap.TreeExplainer(rf)
    sv = explainer.shap_values(Xm)  # shape: (n_obs, n_features)

    shap_store[tcol] = {'sv': sv, 'X': Xm}

    # Top directional finding
    top_i = np.abs(sv).mean(axis=0).argmax()
    high_mask = Xm[:, top_i] > np.median(Xm[:, top_i])
    direction = 'positive (+)' if sv[:, top_i][high_mask].mean() > 0 else 'negative (−)'
    print(f'{tlabel:20s}: top feature = {FEAT_LABELS[top_i]:12s} '
          f'mean|SHAP|={np.abs(sv[:,top_i]).mean():.2f}  '
          f'direction when high = {direction}')

## Figure — SHAP bar plots (mean absolute SHAP values)
Shows the **overall importance** of each feature: mean absolute SHAP value  
averaged across all observations. Higher = more impact on predictions on average.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    ax   = axes[i]
    sv   = shap_store[tcol]['sv']
    mean_abs = np.abs(sv).mean(axis=0)
    ord_b    = np.argsort(mean_abs)

    ax.barh([FEAT_LABELS[j] for j in ord_b], mean_abs[ord_b],
             color=col, alpha=0.82, edgecolor='white', height=0.6)
    for j, v in zip(ord_b, mean_abs[ord_b]):
        ax.text(v + 0.3, list(ord_b).index(j), f'{v:.1f}', va='center', fontsize=9)
    ax.set_xlabel('mean |SHAP value|', fontsize=9)
    ax.set_title(f'{tlabel}', fontsize=10, fontweight='bold')
    ax.set_xlim(0, mean_abs.max() * 1.3)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('SHAP bar plots: mean |SHAP value| per feature\n'
             'Higher = greater mean impact on predictions',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_shap_bar.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_shap_bar.png')

## Figure — SHAP beeswarm plots (directional effects)
Each dot = one observation.  
- **X axis:** SHAP value — positive means the feature pushed prediction UP,  
  negative means it pushed prediction DOWN  
- **Colour:** red = high feature value, blue = low feature value

**How to read it:**  
- Red dots on the right + blue dots on the left → high values increase predictions  
- Red dots on the left + blue dots on the right → high values decrease predictions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 6))
rng = np.random.default_rng(42)

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    ax   = axes[i]
    sv   = shap_store[tcol]['sv']
    Xm   = shap_store[tcol]['X']
    mean_abs = np.abs(sv).mean(axis=0)
    ord_bee  = np.argsort(mean_abs)   # bottom to top order

    for row_i, feat_j in enumerate(ord_bee):
        feat_vals  = Xm[:, feat_j]
        shap_vals  = sv[:, feat_j]

        # Normalise feature values 0→1 for colour mapping
        fmin, fmax = feat_vals.min(), feat_vals.max()
        norm = (feat_vals - fmin) / (fmax - fmin) if fmax > fmin else np.zeros_like(feat_vals)

        # Vertical jitter to separate overlapping points
        jitter = rng.uniform(-0.2, 0.2, len(shap_vals))

        ax.scatter(shap_vals, row_i + jitter,
                   c=norm, cmap='coolwarm', s=18, alpha=0.70, vmin=0, vmax=1)

    ax.set_yticks(range(len(FEAT_LABELS)))
    ax.set_yticklabels([FEAT_LABELS[j] for j in ord_bee], fontsize=8)
    ax.axvline(0, color='#666', lw=0.8, ls='--', alpha=0.6)
    ax.set_xlabel('SHAP value', fontsize=9)
    ax.set_title(f'{tlabel}\n(red=high feature val)', fontsize=9, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('SHAP beeswarm plots — directional feature effects\n'
             'Each point = one observation | Red = high feature value, Blue = low\n'
             'Note: R² negative for all targets; SHAP shows directional associations only',
             fontsize=10, y=1.03)
plt.tight_layout()
plt.savefig('fig_shap_beeswarm.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_shap_beeswarm.png')

## Combined figure (paper version — bar + beeswarm, 2×4)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
rng = np.random.default_rng(42)

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    sv   = shap_store[tcol]['sv']
    Xm   = shap_store[tcol]['X']
    mean_abs = np.abs(sv).mean(axis=0)

    # Top row: bar plot
    ax_bar = axes[0, i]
    ord_b  = np.argsort(mean_abs)
    ax_bar.barh([FEAT_LABELS[j] for j in ord_b], mean_abs[ord_b],
                 color=col, alpha=0.82, edgecolor='white', height=0.6)
    for j, v in zip(ord_b, mean_abs[ord_b]):
        ax_bar.text(v + 0.3, list(ord_b).index(j), f'{v:.1f}', va='center', fontsize=8)
    ax_bar.set_title(f'{tlabel}\nmean |SHAP|', fontsize=9, fontweight='bold')
    ax_bar.set_xlabel('mean |SHAP value|', fontsize=8)
    ax_bar.set_xlim(0, mean_abs.max() * 1.3)
    ax_bar.spines[['top', 'right']].set_visible(False)

    # Bottom row: beeswarm
    ax_bee = axes[1, i]
    ord_bee = np.argsort(mean_abs)
    for row_i, feat_j in enumerate(ord_bee):
        feat_vals = Xm[:, feat_j]
        shap_vals = sv[:, feat_j]
        fmin, fmax = feat_vals.min(), feat_vals.max()
        norm = (feat_vals - fmin)/(fmax - fmin) if fmax > fmin else np.zeros_like(feat_vals)
        jitter = rng.uniform(-0.2, 0.2, len(shap_vals))
        ax_bee.scatter(shap_vals, row_i + jitter,
                       c=norm, cmap='coolwarm', s=18, alpha=0.70, vmin=0, vmax=1)
    ax_bee.set_yticks(range(len(FEAT_LABELS)))
    ax_bee.set_yticklabels([FEAT_LABELS[j] for j in ord_bee], fontsize=8)
    ax_bee.axvline(0, color='#666', lw=0.8, ls='--')
    ax_bee.set_xlabel('SHAP value', fontsize=8)
    ax_bee.set_title(f'{tlabel}\nbeeswarm', fontsize=9)
    ax_bee.spines[['top', 'right']].set_visible(False)

fig.suptitle('SHAP analysis — directional feature effects on RF predictions\n'
             'Top: mean |SHAP| importance | Bottom: beeswarm (red=high feature value)\n'
             'R² negative for all targets (Table 8); SHAP used for directional interpretation only',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('fig_shap_combined.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_shap_combined.png')

In [ ]:
# ── Print key directional findings ────────────────────────────────
print('Key SHAP directional findings (used in paper Discussion 4.2):')
print()
for tcol, (tlabel, col) in TARGETS.items():
    sv   = shap_store[tcol]['sv']
    Xm   = shap_store[tcol]['X']
    mean_abs = np.abs(sv).mean(axis=0)

    # Top 3 features by mean|SHAP|
    top3 = np.argsort(-mean_abs)[:3]
    print(f'{tlabel}:')
    for j in top3:
        feat_vals = Xm[:, j]
        shap_vals = sv[:, j]
        high_mask = feat_vals > np.median(feat_vals)
        mean_high = shap_vals[high_mask].mean()
        direction = 'increases (+)' if mean_high > 0 else 'decreases (−)'
        print(f'  {FEAT_LABELS[j]:12s}: mean|SHAP|={mean_abs[j]:.2f}  '
              f'when high → prediction {direction}')
    print()